# Text Preprocessing - Emotion Dataset
## Minimal Preprocessing for NLP Classification

This notebook performs minimal but effective text preprocessing:
- Lowercase conversion
- Remove URLs, mentions, hashtags
- Remove special characters and digits
- Remove extra whitespaces
- Basic text cleaning

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore')

print("Libraries loaded successfully!")

Libraries loaded successfully!


## 1. Load Raw Data

In [2]:
# Load dataset
df = pd.read_csv('../data/raw/emotion_dataset_v5_clean.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
df.head()

Dataset shape: (79595, 3)

Columns: ['sentence', 'emotion', 'cleaned_text']

First few rows:


,sentence,emotion,cleaned_text
0,I'm so furious with you right now.,anger,i'm so furious with you right now.
1,My day is ruined because of this crap.,anger,my day is ruined because of this crap.
2,I feel like I'm about to explode.,anger,i feel like i'm about to explode.
3,This is the worst day of my life.,anger,this is the worst day of my life.
4,I hate everything about this situation.,anger,i hate everything about this situation.


In [3]:
# Identify text column
text_columns = [col for col in df.columns if col.lower() in ['text', 'sentence', 'content', 'message', 'tweet']]
if text_columns:
    text_col = text_columns[0]
else:
    text_col = [col for col in df.columns if col != 'emotion'][0]

print(f"Text column: '{text_col}'")
print(f"Target column: 'emotion'")
print(f"\nSample texts before preprocessing:")
for i in range(5):
    print(f"{i+1}. {df[text_col].iloc[i]}")

Text column: 'sentence'
Target column: 'emotion'

Sample texts before preprocessing:
1. I'm so furious with you right now.
2. My day is ruined because of this crap.
3. I feel like I'm about to explode.
4. This is the worst day of my life.
5. I hate everything about this situation.


## 2. Preprocessing Functions

In [4]:
def clean_text(text):
    """
    Minimal text cleaning for NLP
    """
    if pd.isna(text):
        return ""
    
    # Convert to string
    text = str(text)
    
    # Lowercase
    text = text.lower()
    
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # Remove mentions and hashtags
    text = re.sub(r'@\w+|#\w+', '', text)
    
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    
    # Remove special characters and digits (keep only letters and spaces)
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # Remove extra whitespaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

print("Preprocessing function defined!")
print("\nTest cleaning function:")
test_text = "OMG!!! I LOVE this 😍 Check it out: https://example.com @user #happy"
print(f"Before: {test_text}")
print(f"After:  {clean_text(test_text)}")

Preprocessing function defined!

Test cleaning function:
Before: OMG!!! I LOVE this 😍 Check it out: https://example.com @user #happy
After:  omg i love this check it out


## 3. Apply Preprocessing

In [5]:
# Apply cleaning
print("Applying text preprocessing...")
df['text_clean'] = df[text_col].apply(clean_text)
print("✓ Preprocessing complete!")

# Show before/after examples
print("\nBefore and After Examples:")
print("="*80)
for i in range(5):
    print(f"\n{i+1}. Original:  {df[text_col].iloc[i]}")
    print(f"   Cleaned:   {df['text_clean'].iloc[i]}")
    print("-"*80)

Applying text preprocessing...
✓ Preprocessing complete!

Before and After Examples:

1. Original:  I'm so furious with you right now.
   Cleaned:   i m so furious with you right now
--------------------------------------------------------------------------------

2. Original:  My day is ruined because of this crap.
   Cleaned:   my day is ruined because of this crap
--------------------------------------------------------------------------------

3. Original:  I feel like I'm about to explode.
   Cleaned:   i feel like i m about to explode
--------------------------------------------------------------------------------

4. Original:  This is the worst day of my life.
   Cleaned:   this is the worst day of my life
--------------------------------------------------------------------------------

5. Original:  I hate everything about this situation.
   Cleaned:   i hate everything about this situation
--------------------------------------------------------------------------------


## 4. Data Quality Check

In [7]:
# Check for empty texts after cleaning
empty_after_clean = (df['text_clean'].str.strip() == '').sum()
print(f"Empty texts after cleaning: {empty_after_clean}")

# Check text length after cleaning
df['clean_length'] = df['text_clean'].str.len()
df['clean_word_count'] = df['text_clean'].str.split().str.len()

print(f"\nText statistics after cleaning:")
print(f"Average length: {df['clean_length'].mean():.2f} characters")
print(f"Average words: {df['clean_word_count'].mean():.2f} words")
print(f"\nMin length: {df['clean_length'].min()} characters")
print(f"Max length: {df['clean_length'].max()} characters")

# Remove empty texts if any
if empty_after_clean > 0:
    print(f"\n Removing {empty_after_clean} empty texts...")
    df = df[df['text_clean'].str.strip() != ''].copy()
    print(f"✓ Dataset shape after removal: {df.shape}")
else:
    print("\n✓ No empty texts found!")

Empty texts after cleaning: 0



Text statistics after cleaning:
Average length: 47.30 characters
Average words: 9.10 words

Min length: 16 characters
Max length: 107 characters

✓ No empty texts found!


## 5. Final Dataset Overview

In [8]:
# Show emotion distribution
print("Emotion distribution after cleaning:")
print(df['emotion'].value_counts())
print(f"\nTotal samples: {len(df):,}")
print(f"Total emotions: {df['emotion'].nunique()}")

Emotion distribution after cleaning:
emotion
happiness         7797
disgust           4315
jealousy          4081
surprise          4062
gratitude         4004
relief            3964
guilt             3961
anger             3860
disappointment    3832
embarrassment     3788
anxiety           3776
pride             3772
hope              3770
loneliness        3751
excitement        3643
fear              3574
sadness           3559
confusion         3429
love              3365
frustration       3292
Name: count, dtype: int64

Total samples: 79,595
Total emotions: 20


In [9]:
# Preview final dataset
print("\nFinal dataset preview:")
df[['text_clean', 'emotion']].head(10)


Final dataset preview:


,text_clean,emotion
0,i m so furious with you right now,anger
1,my day is ruined because of this crap,anger
2,i feel like i m about to explode,anger
3,this is the worst day of my life,anger
4,i hate everything about this situation,anger
5,i m beyond angry i m livid right now,anger
6,my blood is boiling just thinking about it,anger
7,i m fed up with all these lies and excuses,anger
8,this is seriously pissing me off right now,anger
9,i ve had enough of your nonsense already,anger


## 6. Save Processed Data

In [10]:
# Select only necessary columns
df_final = df[['text_clean', 'emotion']].copy()
df_final.columns = ['text', 'emotion']

# Save to processed folder
output_path = '../data/processed/emotion_dataset_preprocessed.csv'
df_final.to_csv(output_path, index=False)

print(f"✓ Preprocessed data saved to: {output_path}")
print(f"  Shape: {df_final.shape}")
print(f"  Columns: {df_final.columns.tolist()}")
print(f"\n{'='*80}")
print("PREPROCESSING COMPLETE!")
print(f"{'='*80}")
print(f"✓ {len(df_final):,} samples ready for modeling")
print(f"✓ {df_final['emotion'].nunique()} emotion classes")
print(f"✓ Average text length: {df_final['text'].str.len().mean():.0f} characters")
print(f"✓ Average word count: {df_final['text'].str.split().str.len().mean():.1f} words")

✓ Preprocessed data saved to: ../data/processed/emotion_dataset_preprocessed.csv
  Shape: (79595, 2)
  Columns: ['text', 'emotion']

PREPROCESSING COMPLETE!
✓ 79,595 samples ready for modeling
✓ 20 emotion classes
✓ Average text length: 47 characters
✓ Average word count: 9.1 words
